In [1]:
import os
from dotenv import load_dotenv
from supabase import create_client
import json
load_dotenv()

# ── supabase ──────────────────────────────────────────────────────────────────
_hf_token      = os.getenv("HF_TOKEN")
_supabase_url  = os.getenv("url")
_supabase_key  = os.getenv("key")

import os

url = os.getenv("url")
key = os.getenv("key")
supabase = create_client(url, key)

from smolagents import CodeAgent, InferenceClientModel, OpenAIModel, tool
import ast
@tool
def query_database(sql: str) -> str:
    """
    Query the PC components database to get prices, benchmarks and compatibility info.
    Args:
        sql: SQL SELECT query to execute (no semicolons).
    """
    try:
        clean_sql = sql.strip().rstrip(";")
        response = supabase.rpc("run_query", {"sql": clean_sql}).execute()
        return json.dumps(response.data, ensure_ascii=False, indent=2) if response.data else "[]"
    except Exception as e:
        return f"Database Error: {e}"

# ── helpers ───────────────────────────────────────────────────────────────────
def get_available_tests() -> list[str]:
    try:
        response = supabase.rpc("run_query", {"sql": "SELECT distinct(test) FROM model_x_test"}).execute()
        return [row["test"] for row in response.data] if response.data else []
    except Exception:
        return ["Relative Performance TechPowerUp"]

def _parse_agent_output(raw) -> dict | str:
    """Пробует привести ответ агента к dict; если не выходит — возвращает как есть."""
    if isinstance(raw, dict):
        return raw
    clean = str(raw).replace("Final answer:", "").strip()
    try:
        return ast.literal_eval(clean)
    except (ValueError, SyntaxError):
        return clean

# ── model / agent factories ───────────────────────────────────────────────────
def make_model(provider: str):
    match provider:
        case "hf" | "huggingface":
            return InferenceClientModel(
                model_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",
                token=_hf_token,
            )
        case "groq":
            return OpenAIModel(
                model_id="llama-3.3-70b-versatile",
                api_base="https://api.groq.com/openai/v1",
                api_key=os.getenv("GROQ_API_KEY"),
            )
        case "google":
            return OpenAIModel(
                model_id="gemini-2.0-flash",
                api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
                api_key=os.getenv("GOOGLE_API_KEY"),
            )
        case _:
            raise ValueError(f"Неизвестный провайдер: {provider}")

def make_agent(model, sys_prompt: str, name: str, max_steps: int = 8) -> CodeAgent:
    return CodeAgent(
        tools=[query_database],
        model=model,
        code_block_tags=("```python", "```"),
        additional_authorized_imports=["json", "re", "pandas"],
        instructions=sys_prompt,
        max_steps=max_steps,
        verbosity_level=0,          # rich-вывод заглушён; логи идут в файл
    )

# бюджетные доли: [min, max] для каждого компонента
RATIO = {
    "GPU":      [0.38, 0.51],
    "CPU_MB":   [0.14, 0.22],
    "RAM":      [0.10, 0.16],
    "PSU":      [0.04, 0.08],
    "STORAGE":  [0.08, 0.16],
    "CASE":     [0.02, 0.06],
    "COOLER":   [0.01, 0.04],
}

if __name__ == "__main__":
    # ── main flow ─────────────────────────────────────────────────────────────────
    available_tests = get_available_tests()
    model_ll = make_model("hf")

C:\Users\ya\Documents\agent_try\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def query_database(sql: str) -> str:
    """
    Query the PC components database to get prices, benchmarks and compatibility info.
    Args:
        sql: SQL SELECT query to execute (no semicolons).
    """
    try:
        clean_sql = sql.strip().rstrip(";")
        response = supabase.rpc("run_query", {"sql": clean_sql}).execute()
        return json.dumps(response.data, ensure_ascii=False, indent=2) if response.data else "[]"
    except Exception as e:
        return f"Database Error: {e}"

response = query_database(
    """
    SELECT
    c.normalized_model_name AS cpu_name,
    t.test,
    t.result,
    c.tdp,
    CASE
        WHEN c.tdp < 70 THEN 'low'
        WHEN c.tdp >= 70 AND c.tdp < 121 THEN 'mid'
        ELSE 'high'
    END AS tdp_tier,
        c.socket,
        c.compatible_chipsets_no_bios_flash
    FROM cpu_x_test AS t
    INNER JOIN cpus AS c
        ON c.id = t.cpu_id
    ORDER BY
        c.normalized_model_name,
        t.test
    LIMIT 8000;
    """
)

In [3]:
converted_dict = json.loads(response) if response != "[]" else {}

In [4]:
response

'[\n  {\n    "tdp": 65,\n    "test": "Cinebench R23 Multi core",\n    "result": 11500,\n    "socket": "AM4",\n    "cpu_name": "AMD Ryzen 5 5600",\n    "tdp_tier": "low",\n    "compatible_chipsets_no_bios_flash": [\n      "A520",\n      "B550",\n      "X570"\n    ]\n  },\n  {\n    "tdp": 65,\n    "test": "Cinebench R23 Single core",\n    "result": 1480,\n    "socket": "AM4",\n    "cpu_name": "AMD Ryzen 5 5600",\n    "tdp_tier": "low",\n    "compatible_chipsets_no_bios_flash": [\n      "A520",\n      "B550",\n      "X570"\n    ]\n  },\n  {\n    "tdp": 65,\n    "test": "CS2",\n    "result": 401,\n    "socket": "AM4",\n    "cpu_name": "AMD Ryzen 5 5600",\n    "tdp_tier": "low",\n    "compatible_chipsets_no_bios_flash": [\n      "A520",\n      "B550",\n      "X570"\n    ]\n  },\n  {\n    "tdp": 65,\n    "test": "PUBG",\n    "result": 145,\n    "socket": "AM4",\n    "cpu_name": "AMD Ryzen 5 5600",\n    "tdp_tier": "low",\n    "compatible_chipsets_no_bios_flash": [\n      "A520",\n      "B550

In [17]:
def get_cpu_mb(min_p, max_p, ddr):
        tmp_sql = f"""
            WITH cpu_x_test_prices AS (
              SELECT
                c.normalized_model_name,
                t.test, t.result,
                p.price_rub,
                p.source_url,
                c.tdp,
                CASE
                    WHEN c.tdp < 70 THEN 'low'
                    WHEN c.tdp >= 70 AND c.tdp < 121 THEN 'mid'
                    ELSE 'high'
                END AS tdp_tier,
                c.socket,
                c.compatible_chipsets_no_bios_flash
              FROM cpu_x_test AS t
              INNER JOIN cpus AS c ON c.id = t.cpu_id
              INNER JOIN component_prices AS p
                  ON c.id = p.component_id
                 AND p.is_available = TRUE
                 AND p.is_verified = TRUE
            ),
            motherboard_prices AS (
              SELECT
                m.normalized_name,
                m.socket,
                m.chipset,
                m.vrm_tier,
                m.form_factor,
                m.ram_type,
                m.num_ram_slots,
                m.cpu_power_pins,
                m.required_cpu_power_pins,
                p.price_rub AS mb_price,
                p.source_url
              FROM motherboards AS m
              INNER JOIN component_prices AS p
                  ON m.id = p.component_id
                 AND p.is_available = TRUE
                 AND p.is_verified = TRUE
            )
            SELECT
              c.normalized_model_name AS cpu_name,
              m.normalized_name AS motherboard_name,
              c.test,
              c.result,
              c.price_rub + m.mb_price AS cpu_and_mb_price,
              c.tdp,
              m.form_factor,
              m.ram_type,
              m.num_ram_slots,
              m.cpu_power_pins,
              m.required_cpu_power_pins,
              m.source_url as motherboard_url,
              c.source_url as cpu_url
            FROM cpu_x_test_prices AS c
            INNER JOIN motherboard_prices AS m
                ON c.socket = m.socket
               AND m.chipset = ANY(c.compatible_chipsets_no_bios_flash)
               AND m.vrm_tier = c.tdp_tier
               AND m.ram_type = '{ddr}'
            WHERE (c.price_rub + m.mb_price) BETWEEN {min_p} AND {max_p}
            ORDER BY (c.price_rub + m.mb_price), cpu_name, motherboard_name, test
            """
        tmp_resp = supabase.rpc("run_query", {"sql": tmp_sql}).execute()
        return tmp_resp.data

get_cpu_mb(10000, 27000, 'DDR4')

[{'tdp': 65,
  'test': 'Cinebench R23 Multi core',
  'result': 12400,
  'cpu_url': 'https://www.wildberries.ru/catalog/363133037/detail.aspx',
  'cpu_name': 'Intel Core i5-12400F',
  'ram_type': 'DDR4',
  'form_factor': 'mATX',
  'num_ram_slots': 2,
  'cpu_power_pins': 8,
  'motherboard_url': 'https://www.wildberries.ru/catalog/171250137/detail.aspx',
  'cpu_and_mb_price': 14134,
  'motherboard_name': 'Gigabyte H610M K DDR4',
  'required_cpu_power_pins': 4},
 {'tdp': 65,
  'test': 'Cinebench R23 Single core',
  'result': 1710,
  'cpu_url': 'https://www.wildberries.ru/catalog/363133037/detail.aspx',
  'cpu_name': 'Intel Core i5-12400F',
  'ram_type': 'DDR4',
  'form_factor': 'mATX',
  'num_ram_slots': 2,
  'cpu_power_pins': 8,
  'motherboard_url': 'https://www.wildberries.ru/catalog/171250137/detail.aspx',
  'cpu_and_mb_price': 14134,
  'motherboard_name': 'Gigabyte H610M K DDR4',
  'required_cpu_power_pins': 4},
 {'tdp': 65,
  'test': 'PUBG',
  'result': 134,
  'cpu_url': 'https://www.

In [5]:
import pandas as pd
df = pd.DataFrame.from_dict(converted_dict)
df.drop(['socket', 'tdp', 'tdp_tier', 'compatible_chipsets_no_bios_flash'], axis=1, inplace=True)

In [6]:
df

,test,result,cpu_name
0,Cinebench R23 Multi core,11500.0,AMD Ryzen 5 5600
1,Cinebench R23 Single core,1480.0,AMD Ryzen 5 5600
2,CS2,401.0,AMD Ryzen 5 5600
3,PUBG,145.0,AMD Ryzen 5 5600
4,Warhammer 40000: Space Marine 2,84.6,AMD Ryzen 5 5600
...,...,...,...
201,Cinebench R23 Single core,2280.0,Intel Core Ultra 7 265KF
202,16 Game FPS Geomean,142.0,Intel Core Ultra 9 285K
203,Cinebench R23 Multi core,41558.0,Intel Core Ultra 9 285K
204,Cinebench R23 Single core,2320.0,Intel Core Ultra 9 285K


In [7]:
df_wide = df.pivot(
    index='cpu_name',
    columns='test',
    values='result'
).reset_index()

# Опционально: убираем имя индекса столбцов, чтобы таблица выглядела чистой
df_wide.columns.name = None

In [22]:
df_wide[['cpu_name', '16 Game FPS Geomean']]

,cpu_name,16 Game FPS Geomean
0,AMD Ryzen 5 5600,115.136968
1,AMD Ryzen 5 5600X,130.884753
2,AMD Ryzen 5 7500F,141.061677
3,AMD Ryzen 5 7600,146.472100
4,AMD Ryzen 5 7600X,147.683831
5,AMD Ryzen 5 9500F,141.442043
6,AMD Ryzen 5 9600,143.321067
7,AMD Ryzen 5 9600X,154.092173
8,AMD Ryzen 7 5700X,117.467087
9,AMD Ryzen 7 5800X,127.605218


In [10]:
df_wide['Cinebench R23 Multi core'] = df_wide['Cinebench R23 Multi core']/100

In [11]:
df_wide['Cinebench R23 Single core'] = df_wide['Cinebench R23 Single core']/100

In [12]:
import pandas as pd
import numpy as np

# Обязательный импорт, без него IterativeImputer не заработает (он пока в режиме experimental)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# --- ШАГ 1: Разделяем метаданные и тесты ---
# Составляем список всех нечисловых столбцов, которые у тебя есть в df_wide
metadata_columns = ['cpu_name', 'tdp', 'tdp_tier', 'socket', 'compatible_chipsets_no_bios_flash']

# Отбираем только те столбцы, которые являются тестами (все остальные)
test_columns = [col for col in df_wide.columns if col not in metadata_columns]

# --- ШАГ 2: Настраиваем и запускаем MICE ---
# max_iter=10 — это количество циклов пересчета (цепочек), 10 обычно за глаза
# random_state=42 — фиксируем сид, чтобы при каждом запуске пропуски заполнялись одинаково
imputer = IterativeImputer(max_iter=10, random_state=42)

# Обучаем импутер и трансформируем только числовую матрицу тестов
# На выходе получаем обычный numpy-массив без названий столбцов
imputed_data = imputer.fit_transform(df_wide[test_columns])

# --- ШАГ 3: Возвращаем данные обратно в DataFrame ---
# Просто перезаписываем столбцы тестов обновленными данными без пропусков
df_wide[test_columns] = imputed_data

In [13]:
df_wide

,cpu_name,16 Game FPS Geomean,CS2,Cinebench R23 Multi core,Cinebench R23 Single core,Cyberpunk 2077,Dragon's Dogma 2,Forza Horizon 5,Ghost of Tsushima,Hogwarts Legacy,...,Kingdom Come: Deliverance 2,Microsoft Flight Simulator,PUBG,Red Dead Redemption 2,S.T.A.L.K.E.R. 2,Silent Hill f,Star Wars Outlaws,Starfield,The Last of Us 2 Remastered,Warhammer 40000: Space Marine 2
0,AMD Ryzen 5 5600,115.136968,401.000000,115.000000,14.800000,102.628115,51.393911,219.210552,163.838256,74.301923,...,71.214149,117.325441,145.000000,170.395918,146.0,96.754763,129.232166,115.976405,78.131778,84.600000
1,AMD Ryzen 5 5600X,130.884753,403.000000,115.000000,15.000000,102.731670,44.600000,219.987233,164.718518,73.939460,...,75.308296,117.791506,145.000000,171.120858,146.0,100.264028,128.610782,115.147882,83.395669,85.000000
2,AMD Ryzen 5 7500F,141.061677,566.444219,141.000000,18.300000,104.743747,69.253456,235.078013,181.821638,66.896981,...,121.199298,126.846544,183.173584,185.205968,146.0,139.599424,116.537770,99.050414,142.399161,103.096510
3,AMD Ryzen 5 7600,146.472100,586.000000,142.000000,18.500000,105.032896,71.270660,237.246659,184.279470,65.884930,...,127.484835,128.147810,188.653002,187.230091,146.0,144.987068,114.802799,96.737103,150.480693,106.000000
4,AMD Ryzen 5 7600X,147.683831,592.000000,143.000000,19.100000,105.112801,72.300000,237.845960,184.958688,65.605251,...,129.446264,128.507416,190.417013,187.789456,146.0,146.668302,114.323341,96.097821,153.002559,108.000000
5,AMD Ryzen 5 9500F,141.442043,593.000000,163.000000,19.800000,104.863914,72.881894,235.979322,182.843187,66.476338,...,131.000000,127.387485,193.038643,186.047295,146.0,148.000000,115.816616,98.088844,155.000000,106.717006
6,AMD Ryzen 5 9600,143.321067,598.000000,163.000000,21.100000,104.943909,73.396276,236.579293,183.523163,66.196348,...,132.757314,127.747489,194.439391,186.607282,146.0,149.506279,115.336625,97.448851,157.259435,107.444546
7,AMD Ryzen 5 9600X,154.092173,604.000000,170.370000,22.440000,105.000000,74.336609,237.000000,184.000000,66.000000,...,138.000000,128.000000,197.483403,187.000000,146.0,154.000000,115.000000,97.000000,164.000000,107.000000
8,AMD Ryzen 7 5700X,117.467087,461.428956,138.000000,15.200000,103.189207,58.506453,223.418829,168.607734,72.338012,...,89.006401,119.850641,154.276843,174.323800,146.0,112.005285,125.865384,111.487315,101.007597,84.000000
9,AMD Ryzen 7 5800X,127.605218,446.000000,154.000000,15.900000,102.853890,61.900000,220.903967,165.757582,73.511602,...,90.187681,118.341786,152.884285,171.976637,146.0,113.017726,127.877232,114.169765,102.526135,86.000000


In [24]:
test_name = '16 Game FPS Geomean'

In [25]:
df_wide[['cpu_name', test_name]]

,cpu_name,16 Game FPS Geomean
0,AMD Ryzen 5 5600,115.136968
1,AMD Ryzen 5 5600X,130.884753
2,AMD Ryzen 5 7500F,141.061677
3,AMD Ryzen 5 7600,146.472100
4,AMD Ryzen 5 7600X,147.683831
5,AMD Ryzen 5 9500F,141.442043
6,AMD Ryzen 5 9600,143.321067
7,AMD Ryzen 5 9600X,154.092173
8,AMD Ryzen 7 5700X,117.467087
9,AMD Ryzen 7 5800X,127.605218


In [30]:
import networkx as nx
import pandas as pd

# 1. Задаем граф зависимостей (направленные ребра от слабых к сильным)
cpu_hierarchy_edges= [
    # Цепочка Intel
    ('Intel Core i5-12400F', 'Intel Core i5-13400F'),
    ('Intel Core i5-12400', 'Intel Core i5-13400'),
    ('Intel Core i5-12400F', 'Intel Core i5-13400'),
    ('Intel Core i5-12400', 'Intel Core i5-13400F'),
    ('Intel Core i5-13400F', 'Intel Core i5-14400F'),
    ('Intel Core i5-14400', 'Intel Core i5-13500'),
    ('Intel Core i5-13500', 'Intel Core i5-14500'),
    ('Intel Core i5-14500', 'Intel Core i5-14600K'),
    ('Intel Core i5-14600K', 'Intel Core i7-14700F'),
    ('Intel Core i7-14700F', 'Intel Core i7-14700KF'),
    ('Intel Core i7-14700KF', 'Intel Core i9-14900KF'),
    ('Intel Core Ultra 5 225', 'Intel Core Ultra 5 245K'),
    ('Intel Core Ultra 5 225F', 'Intel Core Ultra 5 245KF'),
    ('Intel Core Ultra 5 245K', 'Intel Core Ultra 7 265K'),
    ('Intel Core Ultra 5 245KF', 'Intel Core Ultra 7 265KF'),
    ('Intel Core Ultra 7 265K', 'Intel Core Ultra 9 285K'),
    ('Intel Core Ultra 7 265KF', 'Intel Core Ultra 9 285K'),

    # Пересекающиеся цепочки с AMD
    ('AMD Ryzen 5 5600', 'AMD Ryzen 5 5600X'),
    ('AMD Ryzen 5 5600X', 'AMD Ryzen 7 5700X'),
    ('AMD Ryzen 7 5700X', 'AMD Ryzen 7 5800X'),
    ('AMD Ryzen 7 5800X', 'AMD Ryzen 5 5800X3D'),
    ('AMD Ryzen 5 5600X', 'AMD Ryzen 5 7500F'),
    ('AMD Ryzen 5 7500F', 'AMD Ryzen 5 7600'),
    ('AMD Ryzen 5 7600', 'AMD Ryzen 5 7600X'),
    ('AMD Ryzen 5 7600', 'AMD Ryzen 7 7700'),
    ('AMD Ryzen 5 7600X', 'AMD Ryzen 7 7700X'),
    ('AMD Ryzen 5 7600', 'AMD Ryzen 5 9500F'),
    ('AMD Ryzen 5 9500F', 'AMD Ryzen 5 9600'),
    ('AMD Ryzen 5 9600', 'AMD Ryzen 5 9600X'),
    ('AMD Ryzen 5 9600X', 'AMD Ryzen 7 9700X'),
    ('AMD Ryzen 5 7600', 'AMD Ryzen 5 7600X'),
    ('AMD Ryzen 5 7600X', 'AMD Ryzen 9 9900X'),
    ('AMD Ryzen 7 7700X', 'AMD Ryzen 9 9950X'),
    ('AMD Ryzen 5 5600X', 'AMD Ryzen 9 7900X'),
    ('AMD Ryzen 7 5800X', 'AMD Ryzen 9 7950X'),

]
cpu_dag = nx.DiGraph(cpu_hierarchy_edges)
assert nx.is_directed_acyclic_graph(cpu_dag), "Критическая ошибка: в графе иерархии процессоров обнаружен цикл!"


class HierarchicalIterativeImputer:
    """
    Кастомный итеративный импутер, использующий циклическое дообучение моделей
    и коррекцию аномалий на основе направленного графа зависимостей (DAG).
    """
    def __init__(self, dag: nx.DiGraph, constrained_columns: list, max_iter=10, random_state=42):
        self.dag = dag
        self.constrained_columns = constrained_columns
        self.max_iter = max_iter
        self.random_state = random_state
        # Топологическая сортировка гарантирует обход от слабых к сильным
        self.topo_order = list(nx.topological_sort(dag))

    def _apply_constraints(self, df_indexed: pd.DataFrame, cols: list):
        """Применяет жесткие иерархические ограничения к колонкам из списка cols."""
        for col in cols:
            for node in self.topo_order:
                if node not in df_indexed.index:
                    continue

                # Ищем всех непосредственных предшественников (кто слабее текущего процессора)
                predecessors = [p for p in self.dag.predecessors(node) if p in df_indexed.index]

                if predecessors:
                    # Находим максимальный результат среди тех, кто заведомо слабее
                    max_pred_val = df_indexed.loc[predecessors, col].max()
                    current_val = df_indexed.at[node, col]

                    # Если более мощный процессор выдал результат меньше слабого — поднимаем его значение
                    if pd.notna(current_val) and pd.notna(max_pred_val) and current_val < max_pred_val:
                        # Принудительно устанавливаем значение на 1% выше максимального предшественника
                        df_indexed.at[node, col] = max_pred_val * 1.01

    def fit_transform(self, df: pd.DataFrame, test_columns: list, name_col='cpu_name') -> pd.DataFrame:
        df_res = df.copy()

        # 1. Первичное заполнение медианами для старта регрессионных моделей
        for col in test_columns:
            if df_res[col].isna().all():
                df_res[col] = df_res[col].fillna(0.0)
            else:
                df_res[col] = df_res[col].fillna(df_res[col].median())

        # Запоминаем маску, где изначально были NaN
        missing_mask = df[test_columns].isna()

        # Игры, на которые действуют правила графа
        game_cols = [c for c in test_columns if c in self.constrained_columns]

        # Индексируем для удобной работы с графом
        df_res_indexed = df_res.set_index(name_col)

        # Главный цикл чередующихся проекций
        for i in range(self.max_iter):
            # Шаг А: Регрессионная импутация для каждого теста
            for col in test_columns:
                other_cols = [c for c in test_columns if c != col]

                # Обучаемся только на строках, где изначально БЫЛИ реальные данные в базе
                train_idx = df[col].notna()

                if train_idx.sum() > 1: # Нужно хотя бы 2 примера для обучения
                    model = BayesianRidge()
                    X_train = df_res_indexed.loc[train_idx.values, other_cols]
                    y_train = df_res_indexed.loc[train_idx.values, col]

                    model.fit(X_train, y_train)

                    # Предсказываем значения только для тех позиций, где изначально стоял NaN
                    predict_idx = missing_mask[col].values
                    if predict_idx.sum() > 0:
                        X_pred = df_res_indexed.loc[predict_idx, other_cols]
                        predictions = model.predict(X_pred)
                        # Защита от отрицательных FPS/попугаев
                        predictions = np.clip(predictions, a_min=0, a_max=None)
                        df_res_indexed.loc[predict_idx, col] = predictions

            # Шаг Б: Коррекция аномалий по графу (на исправленных данных)
            self._apply_constraints(df_res_indexed, game_cols)

        return df_res_indexed.reset_index()


# -------------------------------------------------------------------------
# ОБНОВЛЕННАЯ ФУНКЦИЯ ЗАПРОСА С НОВЫМ ИМПУТЕРОМ
# -------------------------------------------------------------------------
def get_cpu_tests_pd(test_name: str | None, query_database_func, json_loads_func) -> pd.DataFrame:
    # Запрос данных из базы
    response = query_database_func(
        """
        SELECT
            c.normalized_model_name AS cpu_name,
            t.test,
            t.result
        FROM cpu_x_test AS t
        INNER JOIN cpus AS c
            ON c.id = t.cpu_id
        ORDER BY
            c.normalized_model_name,
            t.test
        LIMIT 8000;
        """
    )

    converted_dict = json_loads_func(response) if response != "[]" else {}
    df = pd.DataFrame.from_dict(converted_dict)

    # Разворачиваем таблицу в wide-формат
    df_wide = df.pivot(
        index='cpu_name',
        columns='test',
        values='result'
    ).reset_index()
    df_wide.columns.name = None

    # Нормализация Cinebench шкал
    if 'Cinebench R23 Multi core' in df_wide.columns:
        df_wide['Cinebench R23 Multi core'] = df_wide['Cinebench R23 Multi core'] / 100
    if 'Cinebench R23 Single core' in df_wide.columns:
        df_wide['Cinebench R23 Single core'] = df_wide['Cinebench R23 Single core'] / 100

    metadata_columns = ['cpu_name']
    test_columns = [col for col in df_wide.columns if col not in metadata_columns]

    # Исключаем синтетику из иерархических правил графа
    cinebench_cols = ['Cinebench R23 Multi core', 'Cinebench R23 Single core']
    constrained_cols = [col for col in test_columns if col not in cinebench_cols]

    # Создаем и запускаем наш кастомный иерархический импутер
    imputer = HierarchicalIterativeImputer(
        dag=cpu_dag,
        constrained_columns=constrained_cols,
        max_iter=10,
        random_state=42
    )

    df_wide = imputer.fit_transform(df_wide, test_columns, name_col='cpu_name')

    if test_name and test_name in df_wide.columns:
        return df_wide[['cpu_name', test_name]]

    return df_wide

In [31]:
print(get_cpu_tests_pd(test_name))

TypeError: get_cpu_tests_pd() missing 2 required positional arguments: 'query_database_func' and 'json_loads_func'